# 6.4 CLAP zero-shot：不更新下游参数的文本分类

本节使用指定的 CLAP（Contrastive Language-Audio Pretraining）checkpoint，
不在 CTMP 训练集上更新参数，而是用六个候选类别的文本提示完成分类。
CLAP预训练本身仍使用音频--文本配对及标签转文本数据。


## 1. 环境准备

执行Notebook前需准备四类资源：
1. CLAP music checkpoint `music_audioset_epoch_15_esc_90.14.pt`（2,352,471,003字节，约2.2 GiB，预先放到`outputs/checkpoints/`）
2. `bert-base-uncased` tokenizer（laion-clap 内部依赖，~700 KB，HF cache）
3. `facebook/bart-base` tokenizer（laion-clap 1.1.7 的导入时依赖，HF cache）
4. `roberta-base` 权重与tokenizer（本节text tower，~480 MB，HF cache）

环境变量约定：
- `HF_ENDPOINT=https://hf-mirror.com`：国内镜像
- `HF_HUB_OFFLINE=1` / `TRANSFORMERS_OFFLINE=1`：Notebook默认只读取本地cache；
  资源缺失时先准备缓存，或在启动Notebook前把二者设为`0`


In [ ]:
import os
os.environ.setdefault('HF_ENDPOINT', 'https://hf-mirror.com')
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')

import sys
from pathlib import Path

# 路径推断：从 cwd 向上找含 CODE/datasets 的目录；PROJECT_ROOT 指向 CODE/
_p = Path.cwd()
while not (_p / "CODE" / "datasets").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/datasets 的目录），请在项目内运行本 Notebook")
    _p = _parent
PROJECT_ROOT = _p / "CODE"  # CODE/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT:', PROJECT_ROOT)


### 1.1 依赖与数据齐备性检查

`laion-clap`和`transformers`是本节额外使用的依赖。CLAP music checkpoint（2,352,471,003字节，约2.2 GiB）和
HF cache（BERT、BART tokenizer及RoBERTa）需预先准备。本单元只检查Python包与CTMP切片是否齐全。


In [ ]:
from chapter06._common import check_environment

check_environment(notebook='06_4_clap', require_ctmp=True)

from transformers import BartTokenizer, BertTokenizer, RobertaTokenizer

for model_name, tokenizer_cls in [
    ('bert-base-uncased', BertTokenizer),
    ('facebook/bart-base', BartTokenizer),
    ('roberta-base', RobertaTokenizer),
]:
    tokenizer_cls.from_pretrained(model_name, local_files_only=True)
    print('local tokenizer cache: OK', model_name)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from chapter06._common import (
    add_recall_colorbar,
    plot_confusion_matrix,
    setup_chinese_font,
)
from chapter06._common.ctmp_loader import (
    CTMP_CLASSES, build_label_map, load_ctmp_segments,
)
from chapter06.pretrained_transformer import (
    CLAPZeroShot, PROMPT_TEMPLATES, render_prompts, INSTRUMENT_EN,
)
from chapter06.pretrained_transformer.run_all_clap import (
    evaluate_encoded_split, upsert_clap_summary,
)

setup_chinese_font()

OUT_DIR = PROJECT_ROOT / 'chapter06' / 'pretrained_transformer' / 'outputs'
FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

LABEL_MAP = build_label_map()
CLASS_NAMES = list(CTMP_CLASSES)
N_CLASSES = len(CLASS_NAMES)
print(f'CTMP 类别: {CLASS_NAMES} ({N_CLASSES} 类)')


## 2. CLAP 架构介绍

CLAP（Contrastive Language-Audio Pretraining）是音频--文本双塔对比学习模型。本节使用LAION-CLAP实现及指定checkpoint：

- **Audio tower**：HTSAT（Hierarchical Token-Semantic Audio Transformer），48 kHz 输入
- **Text tower**：基于 RoBERTa 的文本编码器
- **训练目标**：对批内音频--文本相似度矩阵计算双向对比损失

本章的5秒冻结切片先重采样到48 kHz，并按LAION-CLAP官方示例做int16量化往返；
接口再以`repeatpad`补到480,000个采样点（10秒）。
音频按有界批次编码，避免把整个外部测试集同时送入HTSAT；模型处于评估态，批处理只控制峰值内存。

CLAP论文报告的训练数据包括AudioCaps、Clotho、LAION-Audio-630K和由AudioSet标签生成的文本等。
本节checkpoint文件名为`music_audioset_epoch_15_esc_90.14.pt`，
仅凭文件名不能还原各数据源的精确采样比例。

### 与 AST 的范式差异

| 维度 | AST（判别式） | CLAP（对比式） |
|:---|:---|:---|
| 预训练目标 | 监督分类（AudioSet 527 类） | 对比 (audio, text) 配对 |
| 推理方式 | 经下游训练后输出 logits | 直接 cosine 比较 audio 与 text embedding |
| 下游候选类别 | 分类头输出维度固定 | 推理时由一组类别文本指定 |
| 本章是否更新下游参数 | 是 | 否 |


## 3. prompt 模板设计

### 4 个候选模板

| 模板名 | 模板 | 例子（笛子） |
|:---|:---|:---|
| naive_label | `{name}` | `dizi` |
| recording_of | `a recording of {name}` | `a recording of dizi` |
| descriptive | `the sound of {name}, a Chinese {family}` | `the sound of dizi, a Chinese bamboo flute` |
| domain_scoped | `Chinese traditional music, solo {name} performance` | `Chinese traditional music, solo dizi performance` |

### 类名英文映射（INSTRUMENT_EN）

| 中文 | 英文名 | 乐器族 |
|:---|:---|:---|
| 二胡 | erhu | bowed string instrument |
| 琵琶 | pipa | plucked string instrument |
| 中阮 | zhongruan | plucked string instrument |
| 笛子 | dizi | bamboo flute |
| 唢呐 | suona | double-reed wind instrument |
| 笙 | sheng | free-reed mouth organ |

### 解释边界

不同措辞会改变文本嵌入，因而可能改变zero-shot结果。


In [ ]:
for tname in PROMPT_TEMPLATES:
    print(f'{tname:14s}', render_prompts(tname))


## 4. zero-shot 推理（4 prompts × {test, external_test}）

In [ ]:
clap = CLAPZeroShot()
test_segs = load_ctmp_segments(seed=0, split='test')
ext_segs = load_ctmp_segments(seed=0, split='external_test')
print(f'test={len(test_segs)} ext={len(ext_segs)}')

paths_in = [s['audio_path'] for s in test_segs]
paths_ex = [s['audio_path'] for s in ext_segs]
y_in = np.array([LABEL_MAP[s['family_label']] for s in test_segs])
y_ex = np.array([LABEL_MAP[s['family_label']] for s in ext_segs])
audio_in = clap.encode_audio(paths_in)
audio_ex = clap.encode_audio(paths_ex)

results = {}
for tname in PROMPT_TEMPLATES:
    prompts = render_prompts(tname)
    pred_in, in_acc, _ = evaluate_encoded_split(clap, audio_in, y_in, prompts)
    pred_ex, ex_acc, _ = evaluate_encoded_split(clap, audio_ex, y_ex, prompts)
    results[tname] = {
        'prompts': prompts,
        'y_in': y_in, 'pred_in': pred_in,
        'y_ex': y_ex, 'pred_ex': pred_ex,
    }
    print(f'{tname:14s}  internal acc={in_acc:.3f} | external acc={ex_acc:.3f}')


## 5. prompt × accuracy 主表


In [ ]:
rows = []
for tname, r in results.items():
    rows.append({
        'prompt': tname,
        'internal_acc': accuracy_score(r['y_in'], r['pred_in']),
        'internal_f1':  f1_score(r['y_in'], r['pred_in'], average='macro'),
        'external_acc': accuracy_score(r['y_ex'], r['pred_ex']),
        'external_f1':  f1_score(r['y_ex'], r['pred_ex'], average='macro'),
    })
df_clap = pd.DataFrame(rows)
df_clap.to_csv(OUT_DIR / 'clap_prompt_comparison.csv', index=False, float_format='%.4f')
print('written:', (OUT_DIR / 'clap_prompt_comparison.csv').resolve())
df_clap


## 6. 混淆矩阵：内部 vs 外部（固定展示 descriptive）

固定展示 `descriptive` 模板的两张混淆矩阵，运行时不根据内部或外部测试结果自动选择模板。
四个模板均在主表中报告。


In [ ]:
display_tname = 'descriptive'
print(f'展示 prompt: {display_tname}')
r = results[display_tname]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_confusion_matrix(
    axes[0], r['y_in'], r['pred_in'],
    class_names=CLASS_NAMES, title=f'CLAP · {display_tname} — 内部 test',
    labels=list(range(N_CLASSES)),
)
plot_confusion_matrix(
    axes[1], r['y_ex'], r['pred_ex'],
    class_names=CLASS_NAMES, title=f'CLAP · {display_tname} — 外部 external_test',
    labels=list(range(N_CLASSES)),
)
fig.subplots_adjust(wspace=0.05)
add_recall_colorbar(fig, [axes[0], axes[1]])
fig.savefig(FIG_DIR / 'clap_confusion_internal_vs_external.png', dpi=600, bbox_inches='tight')
plt.show()


## 7. 错误分析

外部 test 上每个真实类别的 top-1 预测分布：哪类乐器最容易被误判，
是被混到哪些类去？这些错误分布可用于提出有关预训练数据覆盖范围和类别描述方式的假设，但仅凭本次结果不能确定具体成因。


In [ ]:
print(f'=== external_test, prompt = "{display_tname}" ===')
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    mask = r['y_ex'] == cls_idx
    if not mask.any():
        continue
    preds = r['pred_ex'][mask]
    counts = pd.Series(preds).value_counts(normalize=True)
    top = counts.head(3)
    top_names = [(CLASS_NAMES[i], f'{p:.2f}') for i, p in top.items()]
    print(f'  {cls_name}: {top_names}')


上述输出只统计六个候选类别之间的误判分布。它可以指出哪些类别文本和音频在当前嵌入空间中更容易混淆，
但不能据此还原CLAP预训练语料的文化分布，也不能确定错误来自文本模板、音频编码、类别边界还是评估样本组成。


## 8. 与 6.1/6.2/6.3/6.4(AST) 全量对比

读 AST notebook 生成的 `pretrained_transformer/outputs/full_comparison.csv`
（已含 6.1b/6.2/6.3/6.4-AST），追加固定的 `descriptive` 行。


In [ ]:
ast_csv = OUT_DIR / 'full_comparison.csv'
if not ast_csv.exists():
    raise FileNotFoundError(
        '缺少AST结果：请先执行06_4_ast_transfer.ipynb，再追加CLAP汇总行'
    )
df_prev = pd.read_csv(ast_csv, dtype=str)

selected = df_clap[df_clap['prompt'] == 'descriptive'].iloc[0]
df_all = upsert_clap_summary(df_prev, selected)
df_all.to_csv(ast_csv, index=False)
print('written:', ast_csv.resolve())
df_all


## 9. 结论

zero-shot 与监督微调采用不同的下游适配方式：
- 本节以候选类别文本做一次冻结评估，不使用下游训练集更新参数
- 不同 prompt 会改变文本嵌入及其与音频嵌入的相似度，因而可能改变分类结果

四个prompt的数值均保留在结果表中，用于描述提示措辞变化时的预测差异。
全章汇总固定追加`descriptive`配置，运行时不依据external_test结果自动改选模板。

CLAP只评估冻结划分0，不报告跨划分均值或标准差。
它与AST等有监督方法的下游适配和统计方式不同。
其分数差异同时关联预训练数据、提示模板、类别边界和评估样本，不能只归因于某一项设计。
